In [1]:
import requests
# import pandas as pd

from datetime import datetime, timezone, timedelta

In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Trabalho/python/projetos/AtualizaIndicador


In [ ]:
# Estaćões das Capitais

url = "https://wis2bra.inmet.gov.br/oapi/collections/stations/items"

todas_estacoes = []

while url:
    resposta = requests.get(url, params={"f": "json", "limit": 1000}, timeout=30)
    resposta.raise_for_status()
    dados = resposta.json()
    todas_estacoes.extend(dados["features"])
    url = next( (link["href"]
                for link in dados.get("links", [])
                if link.get("rel") == "next"),None
              )

estacoes_capitais = {}

for estacao in todas_estacoes:

    prop = estacao["properties"]
    nome = prop["name"].upper()

    for cidade, regiao in capitais_regioes.items():
        if cidade in nome:
            estacoes_capitais[prop["wigos_station_identifier"]] = {"cidade": cidade, "regiao": regiao}


In [3]:

# Todas as observaćões

url = ("https://wis2bra.inmet.gov.br/oapi/collections/urn:wmo:md:br-inmet:synop/items")

agora = datetime.now(timezone.utc)
inicio = agora - timedelta(hours=6)

todas_observacoes = []

parametros = {"f": "json", "limit": 1000, "datetime": f"{inicio.isoformat()}/{agora.isoformat()}"}

while url:
    resposta = requests.get(url, params=parametros, timeout=30)
    resposta.raise_for_status()
    dados = resposta.json()
    todas_observacoes.extend(dados["features"])

    url = next( (link["href"]
                for link in dados.get("links", [])
                if link.get("rel") == "next" ), None
              )

    parametros = None


In [4]:
# Temperaturas por capitais

temperaturas_capitais = {}

for obs in todas_observacoes:
    prop = obs["properties"]

    if prop["name"] != "air_temperature":
        continue

    wigos = prop["wigos_station_identifier"]

    capital = estacoes_capitais.get(wigos)

    if capital is None:
        continue

    cidade = capital["cidade"]

    dt_referencia = datetime.fromisoformat(
        prop["phenomenonTime"].replace("Z", "+00:00")
    )

    nova_temperatura = {
        "regiao": capital["regiao"],
        "cidade": cidade,
        "temperatura": prop["value"],
        "dt_referencia": dt_referencia,
        "status": True
    }

    temperatura_atual = temperaturas_capitais.get(cidade)

    if (
        temperatura_atual is None
        or dt_referencia > temperatura_atual["dt_referencia"]
    ):
        temperaturas_capitais[cidade] = nova_temperatura


# Capitais sem temperatura disponível

for cidade, regiao in capitais_regioes.items():

    if cidade not in temperaturas_capitais:
        temperaturas_capitais[cidade] = {
            "regiao": regiao,
            "cidade": cidade,
            "temperatura": None,
            "dt_referencia": None,
            "status": False
        }

temperaturas = list(temperaturas_capitais.values())


NameError: name 'estacoes_capitais' is not defined

In [ ]:
for temperatura in temperaturas:
    print(temperatura)

In [ ]:
for temperatura in temperaturas:
    if not temperatura["status"]:
        print(temperatura)

In [ ]:

from database.conexao import conectar
from indicadores.temperatura import Temperatura
from datetime import datetime
from decimal import Decimal

with conectar():
    dados_temperatura = Temperatura.buscar()
    print(dados_temperatura)


In [ ]:

from database.conexao import conectar
from indicadores.temperatura import Temperatura

Temperatura.atualizar_temperatura()


In [5]:

from indicadores.estacao import Capital, EstacaoCapital
from database.conexao import conectar

resultado = []

with conectar():

    consulta = (
        EstacaoCapital
        .select(
            EstacaoCapital,
            Capital
        )
        .join(Capital)
        .where(
            EstacaoCapital.status == True
        )
    )

    for estacao in consulta:

        capital = estacao.capital
        ibge = ibge_capitais[capital.uf]

        identificavel = (
            estacao.wigos.startswith(
                f"0-76-0-{ibge}"
            )
        )

        resultado.append({
            "cidade": capital.cidade,
            "uf": capital.uf,
            "estacao": estacao.nome_estacao,
            "wigos": estacao.wigos,
            "identificavel": identificavel
        })
        

NameError: name 'ibge_capitais' is not defined

In [6]:
resultado

[]